# Volatility Dynamics 09 — compound-Poisson jumps

**VD 019.** Module 08 found that volatility-standardized jumps ($|\tilde r|>4$ local $\sigma$) arrive like a homogeneous Poisson process per asset, with ~20% of jump bars shared by several assets. Module 09 describes them as a compound-Poisson process: catalogue, sizes, market-wide vs idiosyncratic arrivals, tick check, and $P(\ge1\text{ jump in }\tau)=1-e^{-\lambda\tau}$.

Detection is Module 08's, unchanged ($c=4$).

**Pre-registered (VD 019 §3):**
- **P1** — a common-jump component exists iff bars with ≥ 2 assets jumping exceed the independence expectation with one-sided Poisson $p<0.01$.
- **P2** — an asset is a tick-discreteness artifact iff its median jump is ≤ 3 index ticks; it is then excluded from Module 10.
- **P3** — GPD vs exponential tail on $z-c$ (reported).
- **P4** — first- vs second-half rate stability (reported, Holm).

Inputs: `state_time`, `underlying`, `return_5m`, `futures_index_price`. Outputs: `09/`.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_jump_compound as jc
START = os.getenv('VD9_START', '').strip() or None
END = os.getenv('VD9_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD9_ASSETS', '').split(',') if x.strip()]

OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '09'
OUT.mkdir(parents=True, exist_ok=True)
print('Module 09', jc.VERSION, jc.selftest())


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

df = load_frame(['futures_index_price'], START, END, ASSETS)
grid = vb.build_grid(df)
res = jc.run(grid)
cat = res['catalog']
print('jumps:', len(cat))
display(res['rates'].round(3))
print(json.dumps(res['cojump'], indent=2, default=float))


In [ ]:
display(res['sizes'].round(3))
display(res['ticks'].round(4))
display(res['stability'].round(4))
display(res['probabilities'][res['probabilities'].tau_hours.isin([4, 8, 16])].round(4))
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(cat['signed_z'], bins=40); axes[0].set_xlabel('signed z (local sigma units)')
axes[1].hist(np.log10(cat['abs_return_bps']), bins=40); axes[1].set_xlabel('log10 |jump| (bps)')
plt.show()


In [ ]:
cat.to_csv(OUT / '09_jump_catalog.csv', index=False)
res['sizes'].to_csv(OUT / '09_jump_sizes.csv', index=False)
res['rates'].to_csv(OUT / '09_arrival_rates.csv', index=False)
res['probabilities'].to_csv(OUT / '09_jump_probability_to_tau.csv', index=False)
res['ticks'].to_csv(OUT / '09_tick_check.csv', index=False)
res['stability'].to_csv(OUT / '09_rate_stability.csv', index=False)
decision = {
    'P1_common_component': res['cojump'],
    'P2_tick_artifact_assets': res['ticks'].loc[res['ticks'].tick_artifact, 'underlying'].tolist(),
    'P3_tail_all': res['sizes'][res['sizes'].underlying == 'ALL'].iloc[0][['gpd_xi', 'gpd_vs_exp_lr_p']].to_dict(),
    'P4_unstable_assets_holm05': res['stability'].loc[res['stability'].holm_p < 0.05, 'underlying'].tolist(),
}
(OUT / '09_decision.json').write_text(json.dumps(decision, indent=2, default=float))
meta = {'version': jc.VERSION, 'module': '09_compound_poisson_jumps', 'c': jc.C,
        'first_time': str(df.state_time.min()), 'last_time': str(df.state_time.max()), 'n_rows': int(len(df)),
        'n_jumps': int(len(cat))}
(OUT / '09_run_metadata.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(decision, indent=2, default=float))
